In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp -r /content/drive/MyDrive/split_dataset_balanced /content/

In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [6]:
train_dir = '/content/split_dataset_balanced/train'
val_dir = '/content/split_dataset_balanced/val'

print(os.path.exists(train_dir), train_dir)
print(os.path.exists(val_dir), val_dir)

True /content/split_dataset_balanced/train
True /content/split_dataset_balanced/val


In [7]:
img_size = 224
batch_size = 32

train_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

In [8]:
train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

class_names = train_ds.classes
print(class_names)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

['destroyed', 'major-damage', 'minor-damage', 'no-damage']
Train samples: 2064
Val samples: 520


In [9]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 4)
)

model = model.to(device)
print(model)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 224MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [11]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1


def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

In [12]:
num_epochs = 5
best_f1 = 0.0
save_path = '/content/best_resnet50_balanced.pth'

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print("-" * 60)

Epoch 1/5
Train Loss: 1.1421 | Train Acc: 0.5223 | Train F1: 0.5236
Val   Loss: 1.0101 | Val   Acc: 0.5981 | Val   F1: 0.5880
------------------------------------------------------------
Epoch 2/5
Train Loss: 0.8191 | Train Acc: 0.6739 | Train F1: 0.6757
Val   Loss: 0.7806 | Val   Acc: 0.6846 | Val   F1: 0.6830
------------------------------------------------------------
Epoch 3/5
Train Loss: 0.7119 | Train Acc: 0.7166 | Train F1: 0.7166
Val   Loss: 0.7443 | Val   Acc: 0.7058 | Val   F1: 0.7000
------------------------------------------------------------
Epoch 4/5
Train Loss: 0.6327 | Train Acc: 0.7544 | Train F1: 0.7546
Val   Loss: 0.8230 | Val   Acc: 0.6904 | Val   F1: 0.6819
------------------------------------------------------------
Epoch 5/5
Train Loss: 0.5548 | Train Acc: 0.7825 | Train F1: 0.7826
Val   Loss: 0.7339 | Val   Acc: 0.6942 | Val   F1: 0.6902
------------------------------------------------------------


In [13]:
model.load_state_dict(torch.load(save_path))

_, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print("Best Validation Accuracy:", val_acc)
print("Best Validation Precision:", precision)
print("Best Validation Recall:", recall)
print("Best Validation Macro F1:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Best Validation Accuracy: 0.7057692307692308
Best Validation Precision: 0.7043489265034507
Best Validation Recall: 0.7057692307692307
Best Validation Macro F1: 0.7000385954506008

Classification Report:

              precision    recall  f1-score   support

   destroyed       0.82      0.92      0.87       130
major-damage       0.63      0.65      0.64       130
minor-damage       0.71      0.52      0.60       130
   no-damage       0.66      0.75      0.70       130

    accuracy                           0.71       520
   macro avg       0.70      0.71      0.70       520
weighted avg       0.70      0.71      0.70       520


Confusion Matrix:

[[119   4   4   3]
 [  9  84  16  21]
 [  8  29  67  26]
 [  9  16   8  97]]


In [14]:
print("\nFINAL METRICS")
print(f"Accuracy  : {val_acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


FINAL METRICS
Accuracy  : 0.7058
Precision : 0.7043
Recall    : 0.7058
F1 Score  : 0.7000
